# Практикум: строим векторную модель для контент-анализа Telegram 

## Шаг 1: выгрузка данных из Telegram-канала 

- извлекаем тексты из Telegram-канала
- сохраняем тексты в формате 'txt' 

In [ ]:
import json

# Загрузка данных
with open("result.json", "r", encoding = "utf-8") as f:
    data = json.load(f)

messages = []

for elem in data["messages"]:
    if type(elem) == dict:
        if type(elem["text"]) == list:
            message = ''
            for el in elem["text"]:
                if type(el) == str:
                    message += el
                else:
                    message += el["text"]
            messages.append(message)
    if type(elem) == str:
        messages.append(elem)


print(messages)

In [ ]:
with open("data.txt", "w", encoding = "utf-8") as f:
    f.writelines(messages)

## Шаг 2: запись текста в переменную и разделение на чанки

In [ ]:
with open("data.txt", "r", encoding = "utf-8") as f:
    texts = f.read()

In [ ]:
texts

Далее - разделение текста на **чанки**. 
Будет использоваться инструмент "рекурсивный сплитер". Этот инструмент создан на основе статистических алгоритмов и построен так, чтобы делить весь текст с учетом особенностей *синтаксиса*. В его основе - рекурсивный алгоритм, который вычисляет повторяющиеся (рекурсивные) паттерны, т.е. закономерности, в тексте.

In [ ]:
# !pip install langchain

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

chunks = text_splitter.split_text(texts)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\n{chunk}\n")

In [ ]:
print(f"Chunk {1}: \n{chunks[0]}\n")
print(len(f"Chunk {1}: \n{chunks[0]}\n"))

## Шаг 3: токенизация

In [ ]:
import re

Напишем функцию для чистки текста, которая:
- приведёт текст к нижнему регистру
- удалит все символы кроме букв
- удалит двойные пробелы

In [ ]:
with open("data.txt", "r", encoding = "utf-8") as f:
    texts = f.readlines()

def clean_text(texts):
    cleaned = []
    for text in texts:
        text = text.lower()
        text = re.sub('[^а-яa-z0-9\s]', '', text)
        text = re.sub(r'\n\n', ' ', text).strip()              
        cleaned.append(text)
    return cleaned

clean_text(texts)

Применяем функцию очистки текста к чанкам:

In [ ]:
new_chunks = []
for chunk in chunks:
    cleaned = clean_text([chunk])
    new_chunks.append(cleaned[0])

for i, chunk in enumerate(new_chunks):
    print(f"Chunk {i+1}:\n{chunk}\n{'-'*40}")

Применяем токенизацию:

In [ ]:
finals = []

for chunk in new_chunks:
    tokenized = chunk.split()
    finals.append(tokenized)

for i, chunk in enumerate(finals):
    print(f'Chunk {i+1}:\n{chunk}\n{"-"*40}')

## Шаг 4: строим модель Word2Vec

In [ ]:
pip install gensim 

In [ ]:
from gensim.models import Word2Vec

# Обучение модели
model = Word2Vec(sentences=finals, vector_size=100, window=5, min_count=1, workers=4)

# Сохранение модели
model.save("word2vec.model")

**Gensim parameters:**

- sentences: this iterable can be simply a list of lists of tokens, but for larger corpora, consider an iterable that streams the sentences directly from disk/network. 
- vector_size: dimensionality of the word vectors.
- window: maximum distance between the current and predicted word within a sentence.
- min_count: ignores all words with total frequency lower than this.
- workers: use these many worker threads to train the model (=faster training with multicore machines).

In [ ]:
# Загрузка модели
model = Word2Vec.load("word2vec.model")

# Получение вектора для слова
vector = model.wv["биоревитализация"]

# Нахождение похожих слов
similar_words = model.wv.most_similar("биоревитализация", topn=3)
print(similar_words)

Оценка векторной близости:

In [ ]:
similarity = model.wv.similarity("фототерапия", "биоревитализация")
print(similarity)

In [ ]:
model.wv['фототерапия']

**Умный вектор** - он же семантический вектор, вектор слова (**word embedding**)— это векторное представление слова, фразы или документа, которое содержит смысловую информацию. Обучается на огромных корпусах текстов и запоминает связи. 

Где применяются "умные" вектора?

- **Информационный поиск**: хранить информацию индекса удобнее в векторах, чтобы найти релевантный ответ в Google Search, достаточно "вытащить" из каталога наиболее близкий по вектору результат

- **Виртуальные ассистенты**: чат-боты на основе ИИ также используют эту технологию, чтобы дополнять свои ответы информацией из векторной базы знаний

- **Машинное обучение**: векторные представления создают перед машинным обучением, чтобы повысить качество классификации или моделирования языка

## Шаг 5: визуализация результатов

**PCA (principal component analysis)** - это статистическая модель, которая поможет нам вычислить из 100 чисел 2 самых важных и привести всю информацию к виду двухмерной матрицы

In [ ]:
pip install matplotlib scikit-learn


In [ ]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')

# Модуль для построения PCA
from sklearn.decomposition import PCA

def pca_scatterplot(model, words=None, sample=0):
  word_vectors = [model.wv[w] for w in words]
  # Сокращаем размерность векторов до 2D
  vectors_2d = PCA().fit_transform(word_vectors)
  # Отрисовка изображения, задаем размер 12 на 10
  plt.figure(figsize=(12,10))
  # Задаем цвет точек и ссылаемся на данные по осям x (0) и y (1)
  plt.scatter(vectors_2d[:,0], vectors_2d[:,1], c='g')
  # Добавляем подписи к данным, проходимся по списку слов
  for i, word in enumerate(words):
    # Соотносим слово с его двухмерным вектором
    plt.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]))

# Картируем несколько слов
pca_scatterplot(model, ['лонгслив', 'illullln', 'карго', 'carne', 'bollente', 
                        'балетки', 'vibram', 'fivefingers', 'five', 
                        'fingers', 'ваша', 'я'])